# LMS-FNet: NIST16 SOTA Evaluation
Domain Adaptation with Data Augmentation.

In [ ]:
import os, gc, io, hashlib, random
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from PIL import Image, ImageChops, ImageEnhance
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor

SEED = 42
tf.keras.utils.set_random_seed(SEED)
print(f'TensorFlow Version: {tf.__version__}')

In [ ]:
class CrossAttentionFusion(layers.Layer):
    def __init__(self, embed_dim=256, num_heads=4, **kwargs):
        super(CrossAttentionFusion, self).__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

    def build(self, input_shape):
        dim = input_shape[0][-1]
        self.Wq_r = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wq_r')
        self.Wk_e = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wk_e')
        self.Wv_e = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wv_e')
        self.Wq_e = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wq_e')
        self.Wk_r = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wk_r')
        self.Wv_r = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wv_r')
        self.Wo = self.add_weight(shape=(self.embed_dim * 2, self.embed_dim), initializer='glorot_uniform', name='Wo')
        self.bias = self.add_weight(shape=(self.embed_dim,), initializer='zeros', name='bias')

    def _scaled_dot_product(self, Q, K, V):
        scale = tf.math.sqrt(tf.cast(self.head_dim, Q.dtype))
        scores = tf.matmul(Q, K, transpose_b=True) / scale
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, V)

    def call(self, inputs):
        raw_feat, ela_feat = inputs
        Q_r = tf.matmul(raw_feat, self.Wq_r)
        K_e = tf.matmul(ela_feat, self.Wk_e)
        V_e = tf.matmul(ela_feat, self.Wv_e)
        Q_e = tf.matmul(ela_feat, self.Wq_e)
        K_r = tf.matmul(raw_feat, self.Wk_r)
        V_r = tf.matmul(raw_feat, self.Wv_r)
        def reshape_heads(x):
            bs = tf.shape(x)[0]
            x = tf.reshape(x, (bs, self.num_heads, self.head_dim))
            return tf.expand_dims(x, axis=2)
        attn_r2e = self._scaled_dot_product(reshape_heads(Q_r), reshape_heads(K_e), reshape_heads(V_e))
        attn_r2e = tf.reshape(attn_r2e, (-1, self.embed_dim))
        attn_e2r = self._scaled_dot_product(reshape_heads(Q_e), reshape_heads(K_r), reshape_heads(V_r))
        attn_e2r = tf.reshape(attn_e2r, (-1, self.embed_dim))
        combined = tf.concat([attn_r2e, attn_e2r], axis=-1)
        return tf.matmul(combined, self.Wo) + self.bias

    def get_config(self):
        config = super().get_config()
        config.update({'embed_dim': self.embed_dim, 'num_heads': self.num_heads})
        return config

class OHEMFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=3.0, alpha=0.25, hard_ratio=0.20, **kwargs):
        super(OHEMFocalLoss, self).__init__(**kwargs)
        self.gamma, self.alpha, self.hard_ratio = gamma, alpha, hard_ratio
    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred, from_logits=False)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        p_t = tf.clip_by_value(p_t, 1e-7, 1.0 - 1e-7)
        alpha_factor = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        focal_loss = alpha_factor * tf.pow(1.0 - p_t, self.gamma) * bce
        bs = tf.shape(focal_loss)[0]
        k = tf.cast(tf.math.ceil(tf.cast(bs, tf.float32) * self.hard_ratio), tf.int32)
        top_k_loss, _ = tf.math.top_k(focal_loss, k=k)
        return tf.reduce_mean(top_k_loss)
    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'alpha': self.alpha, 'hard_ratio': self.hard_ratio})
        return config


## 2. Dynamic Search for NIST16 Dataset

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
CACHE_DIR = Path('/kaggle/working/nist16_cache')

print("Scanning /kaggle/input/ for NIST16 Authentic and Tampered folders...")
input_path = Path('/kaggle/input')
valid_exts = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

au_paths = [str(p) for p in input_path.rglob('*') if any(word in str(p).lower() for word in ['auth', 'pristine', 'world']) and p.is_file() and p.suffix.lower() in valid_exts]
tp_paths = [str(p) for p in input_path.rglob('*') if any(word in str(p).lower() for word in ['tamp', 'probe', 'fake', 'manipulate']) and p.is_file() and p.suffix.lower() in valid_exts]

print(f"Found: {len(au_paths)} Authentic images")
print(f"Found: {len(tp_paths)} Tampered images")

if len(au_paths) == 0 or len(tp_paths) == 0:
    print("WARNING: NIST16 is primarily a localization dataset (tampered images only). If you don't have authentic images attached, this binary classification training will fail! Please ensure your Kaggle upload contains both classes.")
    raise ValueError("Balanced NIST16 dataset not found! Please attach a dataset containing both 'authentic' and 'tampered' folders.")

all_paths = au_paths + tp_paths
all_labels = [1]*len(au_paths) + [0]*len(tp_paths)

paths_train, paths_test, y_train, y_test = train_test_split(
    all_paths, all_labels, test_size=0.20, random_state=SEED, stratify=all_labels
)
y_train = np.array(y_train, dtype=np.int32)
y_test = np.array(y_test, dtype=np.int32)

print(f"Training set: {len(paths_train)} images")
print(f"Testing set:  {len(paths_test)} images")


## 3. Fast Caching and Augmented Dataset Pipeline

In [ ]:
def compute_ela(image_path, quality=91, target_size=(224, 224)):
    img = Image.open(image_path).convert('RGB')
    channels = []
    for q in [75, 85, 95]:
        buf = io.BytesIO()
        img.save(buf, 'JPEG', quality=q)
        buf.seek(0)
        compressed = Image.open(buf)
        ela = ImageChops.difference(img, compressed).convert('L')
        extrema = ela.getextrema()
        max_diff = extrema[1] if isinstance(extrema, tuple) else extrema
        if max_diff == 0: max_diff = 1
        ela = ImageEnhance.Brightness(ela).enhance(255.0 / max_diff)
        channels.append(ela)
    mq_ela = Image.merge('RGB', channels)
    return mq_ela.resize(target_size, Image.LANCZOS)

def prepare_cache(paths, split_name, chunk_size=500):
    raw_cache, ela_cache = [], []
    def _proc(p):
        h = hashlib.md5(p.encode()).hexdigest()[:20]
        rp = CACHE_DIR / split_name / 'raw' / f'{h}.jpg'
        ep = CACHE_DIR / split_name / 'ela' / f'{h}.jpg'
        if not rp.exists() or not ep.exists():
            img = Image.open(p).convert('RGB')
            rp.parent.mkdir(parents=True, exist_ok=True)
            img.resize(IMG_SIZE, Image.LANCZOS).save(rp, 'JPEG', quality=85)
            ep.parent.mkdir(parents=True, exist_ok=True)
            compute_ela(p, target_size=IMG_SIZE).save(ep, 'JPEG', quality=90)
        return str(rp), str(ep)
        
    for start in range(0, len(paths), chunk_size):
        chunk = paths[start:start+chunk_size]
        with ThreadPoolExecutor(max_workers=4) as ex:
            futures = [ex.submit(_proc, p) for p in chunk]
            for f in futures:
                rp, ep = f.result()
                raw_cache.append(rp)
                ela_cache.append(ep)
    return raw_cache, ela_cache

print("Generating NIST16 Cache...")
raw_train, ela_train = prepare_cache(paths_train, 'train')
raw_test, ela_test = prepare_cache(paths_test, 'test')

def load_image(jpeg_path, png_path):
    raw = tf.image.decode_jpeg(tf.io.read_file(jpeg_path), channels=3)
    ela = tf.image.decode_jpeg(tf.io.read_file(png_path), channels=3)
    raw.set_shape([*IMG_SIZE, 3])
    ela.set_shape([*IMG_SIZE, 3])
    return raw, ela

spatial_augmenter = tf.keras.Sequential([layers.RandomFlip("horizontal_and_vertical")])

def augment_batch(data, label):
    raw, ela = data['raw_input'], data['ela_input']
    combined = tf.concat([raw, ela], axis=-1)
    combined = spatial_augmenter(combined, training=True)
    raw_aug, ela_aug = combined[..., :3], combined[..., 3:]
    raw_aug = tf.image.random_brightness(raw_aug, max_delta=0.1)
    raw_aug = tf.image.random_contrast(raw_aug, lower=0.9, upper=1.1)
    return {'raw_input': raw_aug, 'ela_input': ela_aug}, label

def make_dual_ds(raw_paths, ela_paths, labels, training=False, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices((raw_paths, ela_paths, labels))
    def _process(rp, ep, label):
        raw, ela = load_image(rp, ep)
        raw = tf.keras.applications.densenet.preprocess_input(tf.cast(raw, tf.float32))
        ela = tf.keras.applications.densenet.preprocess_input(tf.cast(ela, tf.float32))
        return {'raw_input': raw, 'ela_input': ela}, label
    
    if training:
        ds = ds.shuffle(1024, seed=seed)
        ds = ds.map(_process, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(BATCH_SIZE, drop_remainder=True)
        ds = ds.map(augment_batch, num_parallel_calls=tf.data.AUTOTUNE)
        return ds.prefetch(1)
    else:
        return ds.map(_process, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(1)

train_ds = make_dual_ds(raw_train, ela_train, y_train, training=True)
test_ds = make_dual_ds(raw_test, ela_test, y_test, training=False)


## 4. Stable Domain Adaptation

In [ ]:
strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    model_path = "/kaggle/input/datasets/nikunjkumargond/model1/best_forgery_model_sota.keras"
    if not os.path.exists(model_path):
        model_path = "/kaggle/input/datasets/nikunjkumargond/model1/best_forgery_model.keras"
        
    print(f"Loading Base Model: {model_path}")
    model = tf.keras.models.load_model(
        model_path, 
        custom_objects={'CrossAttentionFusion': CrossAttentionFusion, 'OHEMFocalLoss': OHEMFocalLoss},
        compile=False
    )
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

callbacks = [
    tf.keras.callbacks.ModelCheckpoint('nist16_finetuned.keras', save_best_only=True, monitor='val_auc', mode='max'),
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True)
]

print("Starting Domain Adaptation on NIST16... (20 Epochs)")
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    callbacks=callbacks
)


## 5. Final Evaluation

In [ ]:
print("Running Final Inference on NIST16 test set...")
test_preds = model.predict(test_ds, verbose=0)
y_pred = (test_preds.ravel() >= 0.5).astype(int)
test_acc = np.mean(y_pred == y_test)
test_auc = roc_auc_score(y_test, test_preds)
cm = confusion_matrix(y_test, y_pred, labels=[0,1])

print("\n" + "="*50)
print("NIST16 DOMAIN ADAPTATION RESULTS")
print("="*50)
print(f"Accuracy: {test_acc*100:.2f}%")
print(f"ROC-AUC:  {test_auc:.4f}")
print("Confusion Matrix:")
print(f"True Tampered: {cm[0,0]} | False Authentic: {cm[0,1]}")
print(f"False Tampered: {cm[1,0]} | True Authentic: {cm[1,1]}")
print("="*50)
